# 02 · Run the persona experiment

Every story is run under five conditions, and only the persona changes: no persona, mismatched nationality, matched nationality, full persona (nationality plus sex, age and field of study), and a length-matched persona that pads the matched one with a neutral sentence.

In [2]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/CSS-project

Mounted at /content/drive
/content/drive/MyDrive/CSS-project


In [3]:
!apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, os, time
env = dict(os.environ, OLLAMA_NUM_PARALLEL="4")
subprocess.Popen(["ollama", "serve"], env=env)
time.sleep(5)
!ollama pull llama3.2:3b
!pip install -q openai

Selecting previously unselected package zstd.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.



In [4]:
import json, os, random, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from types import SimpleNamespace
import pandas as pd

## Settings

In [5]:
# MODEL = "gpt-4o-mini"
# BASE_URL = None
# API_KEY = os.environ.get("OPENAI_API_KEY")
MODEL = "llama3.2:3b"
BASE_URL = "http://localhost:11434/v1"
API_KEY = "ollama"
BACKEND = "api"            # api = Ollama API; local = local llama.cpp
# STORIES = "data/stories.csv"
# OUT = Path("results/raw.jsonl")
# LIMIT = None #20                 # pilot first; None = all stories
STORIES = "data_full/stories.csv"
OUT = Path(f"results_full/raw_{MODEL.replace(':', '_')}.jsonl")
LIMIT = None                 # pilot first, then None
K_MISMATCH = 3             # pre-registered: mismatched personas per story
WORKERS = 2 #8
SEED = 2026
args = SimpleNamespace(model=MODEL, backend=BACKEND, seed=SEED)

## Prompts and personas

In [6]:
EMOTIONS = ["joy", "fear", "anger", "sadness", "disgust", "shame", "guilt"]
TASK = ("Here is a situation you experienced:\n\"{text}\"\n\n"
        "What is the main emotion you felt in this situation? Answer with exactly one word "
        "from this list: joy, fear, anger, sadness, disgust, shame, guilt.")
NEUTRAL = ["Answer honestly.", "Please answer honestly.", "Please read this carefully.",
           "Please read the situation carefully.", "Please read the situation below carefully.",
           "Please read the situation below carefully before answering.",
           "Please read the situation below carefully and then answer honestly."]


def nationality_persona(country):
    return f"You are a person from {country}."


def full_persona(row):
    parts = [f"You are a {int(row.age)}-year-old" if pd.notna(row.age) else "You are a",
             row.sex if isinstance(row.sex, str) else "person",
             f"from {row.country}"]
    text = " ".join(parts)
    if isinstance(row.field, str):
        text += f" who studies {row.field}"
    return text + "."


def length_matched_persona(row):
    """Matched nationality persona padded with neutral sentences to the full persona's length."""
    base = nationality_persona(row.country)
    gap = len(full_persona(row).split()) - len(base.split())
    if gap <= 0:
        return base
    filler = min(NEUTRAL, key=lambda s: abs(len(s.split()) - gap))
    return base + " " + filler


def build_jobs(stories):
    countries = sorted(stories.country.unique())
    jobs = []
    for row in stories.itertuples(index=False):
        others = [c for c in countries if c != row.country]
        personas = { "none": "You are a person.",
                    "matched": nationality_persona(row.country),
                    "full": full_persona(row),
                    "length_matched": length_matched_persona(row)}
        # for other in others:
        for other in random.Random(str(row.story_id)).sample(others, k=min(K_MISMATCH, len(others))):
            personas[f"mismatched::{other}"] = nationality_persona(other)
        for cond, persona in personas.items():
            condition, _, persona_country = cond.partition("::")
            if condition in ("matched", "full", "length_matched"):
                persona_country = row.country
            jobs.append({"story_id": row.story_id, "writer_id": row.writer_id,
                         "source_country": row.country, "gold": row.emotion,
                         "condition": condition, "persona_country": persona_country or "",
                         "persona": persona or "", "prompt": TASK.format(text=row.text),
                         "key": f"{row.story_id}|{cond}"})
    return jobs


## Look at the five prompts for one story


In [7]:
stories = pd.read_csv(STORIES)
if LIMIT:
    stories = stories.head(LIMIT)
jobs = build_jobs(stories)
first = stories.story_id.iloc[0]
for j in jobs:
    if j["story_id"] == first:
        print(f'{j["condition"]:<15} ({len(j["persona"].split()):>2} words) {j["persona"] or "(none)"}')
print("\n" + jobs[0]["prompt"])

none            ( 4 words) You are a person.
matched         ( 7 words) You are a person from New Zealand.
full            (11 words) You are a 30-year-old woman from New Zealand who studies languages.
length_matched  (11 words) You are a person from New Zealand. Please read this carefully.
mismatched      ( 6 words) You are a person from Brazil.
mismatched      ( 8 words) You are a person from the United States.
mismatched      ( 6 words) You are a person from Bulgaria.

Here is a situation you experienced:
"The saddest I've ever been was when my lover was killed, hitch-hiking to work."

What is the main emotion you felt in this situation? Answer with exactly one word from this list: joy, fear, anger, sadness, disgust, shame, guilt.


## Calling the model

In [8]:
def call_model(job, args, client):
    if args.backend == "mock":
        rng = random.Random(job["key"])
        return job["gold"] if rng.random() < 0.6 else rng.choice(EMOTIONS)
    messages = ([{"role": "system", "content": job["persona"]}] if job["persona"] else [])
    messages.append({"role": "user", "content": job["prompt"]})
    for attempt in range(6):
        try:
            r = client.chat.completions.create(model=args.model, messages=messages,
                                               temperature=0, max_tokens=30, seed=args.seed)
            return r.choices[0].message.content or ""
        except Exception as e:  # rate limits, timeouts
            wait = 2 ** attempt
            print(f"  retry {attempt + 1} in {wait}s: {e}")
            time.sleep(wait)
    return "__ERROR__"


In [ ]:
# lines = [l for l in OUT.read_text().splitlines() if l.strip() and '"condition": "none"' not in l]
# OUT.write_text("\n".join(lines) + "\n")
# print(len(lines), "rows kept (should be 4224)")

## Run

In [9]:
OUT.parent.mkdir(parents=True, exist_ok=True)
done = set()
if OUT.exists():
    done = {json.loads(l)["key"] for l in OUT.read_text().splitlines() if l.strip()}
todo = [j for j in jobs if j["key"] not in done]
print(f"{len(jobs)} calls in total, {len(done)} already done, {len(todo)} to run.")

client = None
if BACKEND == "api":
    from openai import OpenAI
    client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

with OUT.open("a") as f, ThreadPoolExecutor(WORKERS) as pool:
    futures = {pool.submit(call_model, j, args, client): j for j in todo}
    for i, fut in enumerate(as_completed(futures), 1):
        j = {**futures[fut], "model": MODEL if BACKEND == "api" else "mock", "raw_output": fut.result()}
        j.pop("prompt")
        f.write(json.dumps(j) + "\n"); f.flush()
        if i % 250 == 0:
            print(f"  {i}/{len(todo)}")
print("Done:", OUT)

38556 calls in total, 793 already done, 37806 to run.
  250/37806
  500/37806
  750/37806
  1000/37806
  1250/37806
  1500/37806
  1750/37806
  2000/37806
  2250/37806
  2500/37806
  2750/37806
  3000/37806
  3250/37806
  3500/37806
  3750/37806
  4000/37806
  4250/37806
  4500/37806
  4750/37806
  5000/37806
  5250/37806
  5500/37806
  5750/37806
  6000/37806
  6250/37806
  6500/37806
  6750/37806
  7000/37806
  7250/37806
  7500/37806
  7750/37806
  8000/37806
  8250/37806
  8500/37806
  8750/37806
  9000/37806
  9250/37806
  9500/37806
  9750/37806
  10000/37806
  10250/37806
  10500/37806
  10750/37806
  11000/37806
  11250/37806
  11500/37806
  11750/37806
  12000/37806
  12250/37806
  12500/37806
  12750/37806
  13000/37806
  13250/37806
  13500/37806
  13750/37806
  14000/37806
  14250/37806
  14500/37806
  14750/37806
  15000/37806
  15250/37806
  15500/37806
  15750/37806
  16000/37806
  16250/37806
  16500/37806
  16750/37806
  17000/37806
  17250/37806
  17500/37806
  17750/

## Pilot check

Every output should be a single label from the list. If you see sentences, refusals or other words, fix the prompt before the full run and note the change in the pre-registration.

In [10]:
raw = pd.read_json(OUT, lines=True)
print(raw.raw_output.str.strip().str.lower().value_counts().head(15))
raw.groupby("condition").size()

raw_output
sadness.                                      7102
guilt.                                        5342
shame.                                        5004
fear.                                         4828
anger.                                        4152
joy.                                          3935
disgust.                                      2776
anger                                          550
joy                                            315
disgust                                        196
i would choose the word "shame".               118
i would choose: joy                            103
i would choose: anger.                         102
i would choose the word "guilt".                87
i'd say the main emotion i felt was "joy".      81
Name: count, dtype: int64


,0
condition,
full,5508
length_matched,5508
matched,5508
mismatched,16567
none,5508
